# packages

In [3]:
import os
from glob import glob
import re
import numpy as np
import cv2
from PIL import Image
import argparse
import shutil
import matplotlib.pyplot as plt

# code run only once

## video to frame

In [6]:
def convert_video_to_images(input_video, output_folder):
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        print(f"Creating output folder: {output_folder}")
        os.makedirs(output_folder)
    else:
        print(f"Output folder already exists: {output_folder}")
        shutil.rmtree(output_folder)
        print(f"Recreating output folder: {output_folder}")
        os.makedirs(output_folder)

    # Open the video file
    video_capture = cv2.VideoCapture(input_video)
    success, frame = video_capture.read()
    count = 0

    # Read each frame and save it as an image
    while success: # and count < 1000:
        image_path = os.path.join(output_folder, f"frame_{count:d}.jpg")  # Adjust the format as per your requirement
        cv2.imwrite(image_path, frame)  # Save the frame as an image
        success, frame = video_capture.read()  # Read next frame
        count += 1

    # Release the video capture object
    print(f"Total frames: {count}")
    video_capture.release()

In [7]:
# Call the function to convert the video to images
convert_video_to_images('src/drill/1-1.mp4', 'drill1-1')

Output folder already exists: drill1-1
Recreating output folder: drill1-1
Total frames: 238


In [8]:
convert_video_to_images('src/drill/1-2.mp4', 'drill1-2')

Creating output folder: drill1-2
Total frames: 238


In [9]:
convert_video_to_images('src/drill/1-3.mp4', 'drill1-3')

Creating output folder: drill1-3
Total frames: 237


## functions

In [4]:
def get_mask(frame1, frame2, kernel=np.array((9,9), dtype=np.uint8)):
    """ Obtains image mask
        Inputs: 
            frame1 - Grayscale frame at time t
            frame2 - Grayscale frame at time t + 1
            kernel - (NxN) array for Morphological Operations
        Outputs: 
            mask - Thresholded mask for moving pixels
        """
    frame_diff = cv2.subtract(frame2, frame1)

    # blur the frame difference
    frame_diff = cv2.medianBlur(frame_diff, 3)
    
    mask = cv2.adaptiveThreshold(frame_diff, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,\
                cv2.THRESH_BINARY_INV, 11, 3)

    mask = cv2.medianBlur(mask, 3)

    # morphological operations
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    return mask

def get_contour_detections(mask, thresh=400):
    """ Obtains initial proposed detections from contours discoverd on the mask. 
        Scores are taken as the bbox area, larger is higher.
        Inputs:
            mask - thresholded image mask
            thresh - threshold for contour size
        Outputs:
            detectons - array of proposed detection bounding boxes and scores [[x1,y1,x2,y2,s]]
        """
    # get mask contours
    contours, _ = cv2.findContours(mask, 
                                   cv2.RETR_EXTERNAL, # cv2.RETR_TREE, 
                                   cv2.CHAIN_APPROX_TC89_L1)
    detections = []
    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        area = w*h
        if area > thresh: # hyperparameter
            detections.append([x,y,x+w,y+h, area])
    
    if detections:  # Check if detections list is not empty before conversion
        return np.array(detections)
    else:
        return np.zeros((0, 5)) #np.empty((0, 5))  # Return an empty array if no detections meet the criteria
    #return np.array(detections)


# =============================================================================
# Non-Max Supression for detected bounding boxes on blobs

def remove_contained_bboxes(boxes):
    """ Removes all smaller boxes that are contained within larger boxes.
        Requires bboxes to be soirted by area (score)
        Inputs:
            boxes - array bounding boxes sorted (descending) by area 
                    [[x1,y1,x2,y2]]
        Outputs:
            keep - indexes of bounding boxes that are not entirely contained 
                   in another box
        """
    check_array = np.array([True, True, False, False])
    keep = list(range(0, len(boxes)))
    for i in keep: # range(0, len(bboxes)):
        for j in range(0, len(boxes)):
            # check if box j is completely contained in box i
            if np.all((np.array(boxes[j]) >= np.array(boxes[i])) == check_array):
                try:
                    keep.remove(j)
                except ValueError:
                    continue
    return keep


def non_max_suppression(boxes, scores, threshold=1e-1):
    """
    Perform non-max suppression on a set of bounding boxes 
    and corresponding scores.
    Inputs:
        boxes: a list of bounding boxes in the format [xmin, ymin, xmax, ymax]
        scores: a list of corresponding scores 
        threshold: the IoU (intersection-over-union) threshold for merging bboxes
    Outputs:
        boxes - non-max suppressed boxes
    """
    # Sort the boxes by score in descending order
    boxes = boxes[np.argsort(scores)[::-1]]

    # remove all contained bounding boxes and get ordered index
    order = remove_contained_bboxes(boxes)

    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        for j in order:
            # Calculate the IoU between the two boxes
            intersection = max(0, min(boxes[i][2], boxes[j][2]) - max(boxes[i][0], boxes[j][0])) * \
                           max(0, min(boxes[i][3], boxes[j][3]) - max(boxes[i][1], boxes[j][1]))
            union = (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1]) + \
                    (boxes[j][2] - boxes[j][0]) * (boxes[j][3] - boxes[j][1]) - intersection
            iou = intersection / union

            # Remove boxes with IoU greater than the threshold
            if iou > threshold:
                order.remove(j)
                
    return boxes[keep]


def get_detections(frame1, frame2, bbox_thresh=400, nms_thresh=1e-3, mask_kernel=np.array((9,9), dtype=np.uint8)):
    """ Main function to get detections via Frame Differencing
        Inputs:
            frame1 - Grayscale frame at time t
            frame2 - Grayscale frame at time t + 1
            bbox_thresh - Minimum threshold area for declaring a bounding box 
            nms_thresh - IOU threshold for computing Non-Maximal Supression
            mask_kernel - kernel for morphological operations on motion mask
        Outputs:
            detections - list with bounding box locations of all detections
                bounding boxes are in the form of: (xmin, ymin, xmax, ymax)
        """
    # get image mask for moving pixels
    mask = get_mask(frame1, frame2, mask_kernel)

    # get initially proposed detections from contours
    detections = get_contour_detections(mask, bbox_thresh)

    # separate bboxes and scores
    bboxes = detections[:, :4]
    scores = detections[:, -1]
    
    if bboxes.size > 0:
        return bboxes
    else:
        # perform Non-Maximal Supression on initial detections
        return non_max_suppression(bboxes, scores, nms_thresh)


  
kernel=np.array((9,9), dtype=np.uint8)

In [5]:
def draw_bboxes(frame, detections):
    for det in detections:
        x1,y1,x2,y2 = det
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 3)


def create_gif_from_images(save_path : str, image_path : str, ext : str) -> None:
    ''' creates a GIF from a folder of images
        Inputs:
            save_path - path to save GIF
            image_path - path where images are located
            ext - extension of the images
        Outputs:
            None
    '''
    print('start to create GIF')
    ext = ext.replace('.', '')
    image_paths = sorted(glob(os.path.join(image_path, f'*.{ext}')))
    image_paths.sort(key=lambda f: int(''.join(filter(str.isdigit, f))))
    pil_images = [Image.open(im_path) for im_path in image_paths]

    pil_images[0].save(save_path, format='GIF', append_images=pil_images,
                       save_all=True, duration=50, loop=0)

The GIF of benchmark video (1-1)

In [10]:
create_gif_from_images('src.GIF', 'drill1-1', '.jpg')

start to create GIF


In [6]:
def fd(src, new):
    # src: name of folder
    # new: name of folder
    src_paths = sorted(glob(f"{src}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    new_paths = sorted(glob(f"{new}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    
    thresh = 1000
    num_frames = 237
    box_result = []
    for idx in range(0, num_frames):
        # read frames
        frame1_bgr = cv2.imread(src_paths[idx])
        frame2_bgr = cv2.imread(new_paths[idx])

        # get detections
        detections = get_detections(cv2.cvtColor(frame1_bgr, cv2.COLOR_BGR2GRAY), 
                                    cv2.cvtColor(frame2_bgr, cv2.COLOR_BGR2GRAY), 
                                    bbox_thresh=thresh,
                                    nms_thresh=1e-4)
        box_result.append(detections)
    print('the box thresh: ', thresh)
    
    ####visualize
    if not os.path.exists('temp'):
        os.makedirs('temp')
    else:
        shutil.rmtree('temp')
        os.makedirs('temp')
        
    print('start to visualize the box on the images')

    for idx in range(0, num_frames):
        # read frames
        frame_bgr = cv2.imread(new_paths[idx])
        detections = box_result[idx]                           
        # draw bounding boxes on frame
        draw_bboxes(frame_bgr, detections)

        # save image for GIF
        fig = plt.figure(figsize=(1280/100, 720/100)) # (15, 7)  / (1280/100, 720/100)
        plt.imshow(frame_bgr)
        plt.axis('off')
        fig.savefig(f"temp/frame_{idx}.png")
        plt.close()

    file_path = f"{new}.GIF"

    # Check if the file exists before attempting to delete it
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

    create_gif_from_images(file_path, 'temp', '.png')

# code to execute

In [7]:
fd('drill1-1', 'drill1-2') #the result is saved in drill1-3.GIF
fd('drill1-1', 'drill1-3') #the result is saved in drill1-3.GIF

the box thresh:  1000
start to visualize the box on the images
drill1-2.GIF has been deleted.
start to create GIF
